# OpenCode AI Gateway — Google Colab Notebook

This notebook sets up and runs the **OpenCode OpenAI-Compatible Rust AI Gateway** (https://github.com/Subham460/ai-gateway) entirely inside Colab.

**What it does:**
- Installs the Rust toolchain and Bun (runtime deps of the gateway)
- Clones the gateway repo and builds the release binary
- Starts the gateway and runs smoke tests (health, models, chat completion, streaming)
- (Optional) Runs the built-in integration test suite and exposes the gateway via a tunnel

> Note: the gateway talks to OpenCode's hosted inference upstream — you do **not** need the OpenCode CLI installed for it to work.

**Run the cells top to bottom.** Build takes ~3–6 min on first run per session.

---


## 1. Install Rust toolchain

In [ ]:
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs \
    | sh -s -- -y --profile minimal > /dev/null 2>&1
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ["PATH"]
!cargo --version


## 2. Install Bun (runtime dep for upstream calls)

In [ ]:
!curl -fsSL https://bun.sh/install | bash > /dev/null 2>&1
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.bun/bin:" + os.environ["PATH"]
!bun --version


## 3. Install OpenCode CLI (the CLI agent)

In [ ]:
!curl -fsSL https://opencode.ai/install | bash
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.opencode/bin:" + os.environ["PATH"]
!opencode --version


## 4. Clone the gateway repo

In [ ]:
import os
if not os.path.isdir('/content/ai-gateway'):
    !git clone https://github.com/Subham460/ai-gateway.git /content/ai-gateway
print('Repo ready at /content/ai-gateway')


## 5. Build the release binary (first time only)

In [ ]:
!cd /content/ai-gateway && cargo build --release 2>&1 | tail -5
!ls -lh /content/ai-gateway/target/release/opencode-ai-gateway


## 5b. (Alternative) Download the prebuilt release binary instead of building

Skip section 5 if you prefer — this downloads the official Linux x86-64 binary from the GitHub Release, so no compile is needed.

In [ ]:
import os
os.makedirs('/content/ai-gateway/target/release', exist_ok=True)
!wget -q https://github.com/Subham460/ai-gateway/releases/download/v0.1.0/opencode-ai-gateway \
  -O /content/ai-gateway/target/release/opencode-ai-gateway
!chmod +x /content/ai-gateway/target/release/opencode-ai-gateway
!ls -lh /content/ai-gateway/target/release/opencode-ai-gateway


## 6. Start the gateway

In [ ]:
import os, subprocess, time

PORT = "18080"   # avoid Colab's busy 8080
BIN = "/content/ai-gateway/target/release/opencode-ai-gateway"

os.environ["BUN_PATH"] = f"{os.path.expanduser('~')}/.bun/bin/bun"
os.environ["PORT"] = PORT

# Kill any previous instance before starting a new one
subprocess.run(["pkill", "-f", "opencode-ai-gateway"], capture_output=True)
time.sleep(1)

proc = subprocess.Popen([BIN], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
time.sleep(2)
print("Gateway PID:", proc.pid, "| listening on port", PORT)


## 7. Smoke test — health & model catalog

In [ ]:
print('== HEALTH ==')
!curl -s http://localhost:18080/health
print('\n== MODELS (first 3) ==')
!curl -s http://localhost:18080/v1/models | python3 -c "import sys,json; d=json.load(sys.stdin); [print(m['id']) for m in d['data'][:3]]"


## 8. Smoke test — non-streaming chat completion

In [ ]:
!curl -s http://localhost:18080/v1/chat/completions \
  -H "Content-Type: application/json" -H "Authorization: Bearer sk-any" \
  -d '{"model":"deepseek-v4.1-flash-free","messages":[{"role":"user","content":"Say hello in one short sentence."}]}'


## 9. Smoke test — streaming (SSE)

In [ ]:
!curl -N http://localhost:18080/v1/chat/completions \
  -H "Content-Type: application/json" -H "Authorization: Bearer sk-any" \
  -d '{"model":"deepseek-v4.1-flash-free","stream":true,"messages":[{"role":"user","content":"Count from 1 to 3."}]}' | head -c 2000
print('\n(Done)')


## 10. (Recommended) Integration test suite — uses a mock upstream

This validates the whole gateway with a mock upstream on port 18099, so it works even when the live free-tier quota is unavailable.

In [ ]:
!OPENCODE_INFERENCE_URL=http://127.0.0.1:18099/chat/completions PORT=18093 \
  /content/ai-gateway/target/release/opencode-ai-gateway > /tmp/gw_test.log 2>&1 &
!sleep 1
!GATEWAY_PORT=18093 python3 /content/ai-gateway/tests/test_gateway.py
subprocess.run(["pkill", "-f", "opencode-ai-gateway"], capture_output=True)
print('Test gateway stopped.')


## 11. (Optional) Expose the gateway to your laptop

Colab only exposes the gateway on localhost inside the VM. Use a tunnel to reach it from your browser/OpenAI SDK on your own machine.

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
url = ngrok.connect(18080)
print('Your OpenAI base_url:', str(url) + '/v1')
print('Example:  client = OpenAI(base_url="' + str(url) + '/v1", api_key="sk-any")')


## 12. Cleanup — stop the gateway

In [ ]:
import subprocess
subprocess.run(["pkill", "-f", "opencode-ai-gateway"], capture_output=True)
print('Gateway stopped.')
